# Libras Livre — treino em GPU (Colab ou Kaggle)

Roda o mesmo código de `computer-vision-model/treino/` numa GPU gratuita. O que muda
em relação a rodar no notebook local é só a velocidade: no CPU cada rodada da
leave-one-signer-out leva ~20 min; na GPU, minutos.

**Por que isso vale a pena:** os experimentos que faltam (GCN vs ResNet, pré-treino na
V-LIBRASIL, features de ângulo, mais épocas) são uma rodada de treino cada. No CPU
são horas e o notebook fica a 93°C; aqui são minutos e a máquina fica livre.

**O que NÃO se move para cá:** a extração de landmarks. Ela é MediaPipe em CPU, dura
horas e GPU não acelera — continua rodando na máquina local.

---

## Antes de começar

1. **Ative a GPU:** Colab → *Ambiente de execução > Alterar tipo de ambiente > GPU*.
   Kaggle → painel direito, *Accelerator > GPU*.
2. **Tenha o `landmarks.tar.gz`** (~44 MB) à mão. Gere na máquina local com:
   ```bash
   cd computer-vision-model
   tar czf landmarks.tar.gz -C PoC/data landmarks
   ```

## 1. Ambiente e código

In [ ]:
import os, pathlib, subprocess, sys

EM_COLAB = "google.colab" in sys.modules or os.path.exists("/content")
EM_KAGGLE = os.path.exists("/kaggle/working")
BASE = pathlib.Path("/content" if EM_COLAB else "/kaggle/working" if EM_KAGGLE else ".")
print("ambiente:", "Colab" if EM_COLAB else "Kaggle" if EM_KAGGLE else "local", "| base:", BASE)

# A branch do trabalho de modelo. O clone JÁ aponta para ela: um `clone --depth 1`
# sem --branch traz a default (main), que não tem computer-vision-model/treino/,
# e o erro só apareceria lá na frente como "arquivo não encontrado".
BRANCH = "claude/libras-detection-model-53kd30"
REPO = BASE / "libras-livre-ai-glasses-brasil"
if not REPO.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH,
                    "https://github.com/Heitorvazeg/libras-livre-ai-glasses-brasil.git",
                    str(REPO)], check=True)

TREINO = REPO / "computer-vision-model" / "treino"
# Falha alto e agora, não daqui a três células: se o clone trouxe a branch errada,
# é aqui que se descobre.
assert TREINO.is_dir(), f"{TREINO} não existe — o clone trouxe a branch certa?"
print("código em:", TREINO)
print("branch:", subprocess.run(["git", "-C", str(REPO), "rev-parse", "--abbrev-ref", "HEAD"],
                                capture_output=True, text=True).stdout.strip())


In [ ]:
import torch
print("torch", torch.__version__, "| CUDA disponível:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("!! sem GPU — ative o acelerador antes de treinar, senão isto vira CPU lento")

# pyyaml costuma já existir nos dois ambientes; torch/torchvision também.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml"], check=False)

## 2. Landmarks

Escolha **um** dos caminhos abaixo. O destino é sempre
`computer-vision-model/PoC/data/landmarks/`, que é onde `treinar.py` procura.

In [ ]:
import shutil, tarfile

DESTINO = REPO / "computer-vision-model" / "PoC" / "data"
DESTINO.mkdir(parents=True, exist_ok=True)

# --- Caminho A: upload manual (funciona nos dois ambientes) ---
if EM_COLAB:
    from google.colab import files
    print("selecione o landmarks.tar.gz")
    enviados = files.upload()
    origem = pathlib.Path(next(iter(enviados)))
else:
    # Kaggle: adicione o .tar.gz como Dataset e ajuste o caminho abaixo.
    origem = pathlib.Path("/kaggle/input/libras-landmarks/landmarks.tar.gz")

# --- Caminho B (Colab): montar o Drive e apontar para o arquivo lá ---
# from google.colab import drive; drive.mount('/content/drive')
# origem = pathlib.Path('/content/drive/MyDrive/libras/landmarks.tar.gz')

with tarfile.open(origem) as tar:
    tar.extractall(DESTINO)

n = len(list((DESTINO / "landmarks").glob("*.npy")))
print(f"{n} arquivos de landmarks em {DESTINO / 'landmarks'}")
assert n > 0, "nada extraído — confira o caminho do .tar.gz"

## 3. Treino

`--dispositivo auto` pega a GPU sozinho. As duas arquiteturas usam exatamente o mesmo
protocolo (leave-one-signer-out, uma pessoa inteira fora por rodada), então os números
são comparáveis entre si e com o baseline DTW da PoC (70,0%).

In [ ]:
%cd {TREINO}
# Validação rápida do encanamento (segundos, dados sintéticos).
!python selftest.py

In [ ]:
# ST-GCN — grafo do esqueleto, 0,46M parâmetros
!python treinar.py --arquitetura gcn --dispositivo auto --batch 64

In [ ]:
# Skeleton-DML + ResNet-18 — para reproduzir/comparar na mesma GPU
!python treinar.py --arquitetura resnet --dispositivo auto --batch 64

## 4. Resultados

Os relatórios ficam em `resultados-gcn/relatorio.md` e `resultados-resnet/relatorio.md`.
**Baixe-os antes de fechar a sessão** — o disco do Colab/Kaggle é descartado ao encerrar.

In [ ]:
for arq in ("gcn", "resnet"):
    rel = TREINO / f"resultados-{arq}" / "relatorio.md"
    if rel.exists():
        print("=" * 70)
        print(rel.read_text(encoding="utf-8")[:1500])

if EM_COLAB:
    from google.colab import files
    for arq in ("gcn", "resnet"):
        rel = TREINO / f"resultados-{arq}" / "relatorio.md"
        if rel.exists():
            files.download(str(rel))